# 🌾 KisanScore ML Engine: Model Training & SHAP Explainability

This notebook covers the end-to-end Machine Learning pipeline for **KisanScore**:
1. **Data Ingestion & EDA**: Loading agricultural indicators (`ndvi_5yr_avg`, `rainfall_mm`, `soil_npk_index`, `crop_price_inr`, `farmer_type`).
2. **Preprocessing Pipeline**: Handling Cold-Start (New Farmers vs Experienced Farmers), categorical encoding, and feature preparation.
3. **Model Training & Comparison**: Benchmarking Logistic Regression, Random Forest, and **XGBoost Classifier**.
4. **Comprehensive Evaluation**: 5-Fold Stratified Cross-Validation, Subgroup Fairness (Cold-Start vs Historical), ROC-AUC, and Confusion Matrix.
5. **SHAP Explainability**: Translating feature attributions into human-friendly impact cards for the Bank Officer dashboard.
6. **Artifact Export & API Inference**: Testing the production endpoint payload structure.

In [ ]:
# Step 1: Import Required Libraries
import os
import json
import numpy as np
import pandas as pd
import joblib

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from xgboost import XGBClassifier
import shap

print("All libraries successfully imported!")

## 📊 Step 2: Data Loading & Exploratory Data Analysis

In [ ]:
df = pd.read_csv('dataset.csv')
print(f"Dataset Shape: {df.shape}")
display(df.head(10))
print("\nSummary Statistics:")
display(df.describe())
print("\nTarget Class Distribution (default_risk_label):")
print(df['default_risk_label'].value_counts(normalize=True))

## 🛠️ Step 3: Feature Engineering & Preprocessing Pipeline

In [ ]:
from preprocess import load_and_preprocess_data, FEATURE_COLUMNS

X_train, X_test, y_train, y_test, X_all, y_all = load_and_preprocess_data()
print(f"Features used for training: {FEATURE_COLUMNS}")
print(f"Training set size: {X_train.shape[0]} | Test set size: {X_test.shape[0]}")
display(X_train.head())

## 🚀 Step 4: Model Training & Benchmarking

In [ ]:
# Model Definitions
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost Classifier": XGBClassifier(
        n_estimators=120,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42,
        use_label_encoder=False
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probas = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probas)
    f1 = f1_score(y_test, preds)
    results[name] = {"Accuracy": acc, "ROC-AUC": auc, "F1-Score": f1}
    print(f"{name:20s} -> Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f} | F1-Score: {f1:.4f}")

xgb_model = models["XGBoost Classifier"]

## 📈 Step 5: Model Evaluation & Classification Report

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
print("=== Classification Report (XGBoost) ===")
print(classification_report(y_test, y_pred_xgb, target_names=["Low Default Risk (0)", "High Default Risk (1)"]))

print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred_xgb)
print(cm)

## 🔍 Step 6: SHAP Explainability & Credit Score Mapping

In [ ]:
# Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print("SHAP values calculated successfully!")
# Summary Plot
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLUMNS, show=False)

## 🌐 Step 7: Testing API Production Endpoint & Inference

In [ ]:
from inference import get_application_score

# Sample 1: Cold-Start Applicant (New Farmer)
cold_start_applicant = {
    "farmer_type": "New_Farmer",
    "ndvi_5yr_avg": 0.0,
    "rainfall_mm": 650.0,
    "soil_npk_index": 68.5,
    "crop_price_inr": 2450.0
}

res_cold = get_application_score("APP_9942", cold_start_applicant)
print("=== Result for Route A (Cold-Start Farmer) ===")
print(json.dumps(res_cold, indent=2))

# Sample 2: Experienced Applicant (Historical NDVI)
exp_applicant = {
    "farmer_type": "Experienced",
    "ndvi_5yr_avg": 0.82,
    "rainfall_mm": 820.0,
    "soil_npk_index": 84.0,
    "crop_price_inr": 2850.0
}

res_exp = get_application_score("APP_9943", exp_applicant)
print("\n=== Result for Route B (Fast-Track Experienced Farmer) ===")
print(json.dumps(res_exp, indent=2))